## 🔰 PyTorchでニューラルネットワーク基礎　#38 【GPT編・事前学習】


### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください

### データについて
* Livedoorニュースコーパスのlivedoor-hommeカテゴリを利用します。
* huggingfaceの　llm-book/livedoor-news-corpus　などから適宜ダウンロードしてください。
* 先頭から3行のURL、日付、タイトルを削除したものを利用します。

### トークナイザーについて
* tokenizer/livedoor_homme_tokenizer_8k.json
    * bytelevel BPEで構成した語彙数8kのtokenizer


### 今回扱う内容
1. GPTタイプの事前学習
2. Datasetクラスのカスタマイズ
3. Flash Attention
4. 日本語データでの学習

### 注意点
* 汎用的な内容は生成不能
* 学習した内容、学習した文と類似文章が入力されるとうまく生成される
* 学習していない内容は、全くだめ
* モデルサイズ・データサイズが小さいので完全コピーに近い生成がみられるがこれが正常な状況
* 系列長を512とある程度長くしないと、指示チューニング時にトークン不足となる
* 事前学習だけを考える場合、GPUメモリとの兼ね合いで seq_len=128でも事前学習による生成可能

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from pathlib import Path


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

device=device(type='cuda')


In [ ]:
# ディレクトリーは必要に応じて変更してください
data_dir = "./data/homme/"                                          # 学習用データのディレクトリ
tokenizer_filename = "tokenizer/livedoor_homme_tokenizer_8k.json"   # tokenizer
pretrain_filename = "model/homme_seq_512_bpe_8k.model"

#--- トークナイザー
tokenizer = Tokenizer.from_file(tokenizer_filename)
print(f"size: {tokenizer.get_vocab_size()}")

size: 8000


## GPTタイプモデル

In [ ]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.seq_len = 512   # 128トークンだとSFT時に少ないが文章生成だけなら問題ない
        self.d_model = 256   # 精度が悪い場合は 512に変更
        self.nhead = 8
        self.dim_feedforward = 4*self.d_model
        self.num_layers = 6
        self.dropout = 0.1
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.eod_token_id = tokenizer.token_to_id("<eod>")
        
        # 学習データに関する設定
        self.context_size = self.seq_len         # 学習できる長さ
        self.context_stride = self.context_size  # 重なり具合の調整
        self.samples_per_epoch=10_000            # 1epochのデータ数、ランダム開始位置の個数

        # 学習設定
        self.batch_size = 128       # out of memoryの場合、バッチサイズを小さくする
        self.learning_rate = 0.001  # デフォルトと変わらない
        self.num_steps = 5500       # 繰り返しステップ数
        self.max_grad_norm = 1.0


In [ ]:
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        
        # 埋め込み層 pad id 学習しない(今回ないはずだが)
        self.token_embedding = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim=config.d_model, padding_idx=config.pad_token_id)
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        self.dropout = nn.Dropout(config.dropout)

        # Transformer layers
        # TransformerEncoderだけど、右三角にmaskつけるのでマスク付き自己注意のタイプになる
        causal_transformer_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True     # 正規化の場所指定
        )
        self.transformer = nn.TransformerEncoder(causal_transformer_layer, num_layers=config.num_layers, enable_nested_tensor=False)
        
        # 最後の出力に向けた正規化とFC層　最終的に単語数になる
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.fc = nn.Linear(config.d_model, config.vocab_size,bias=False)
        self.apply(self._init_weights)     # 埋め込み部分の初期重み変更
        self.fc.weight = self.token_embedding.weight      # 重み共有

    # 埋め込み部分の初期化GPT2タイプ
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len,device=x.device)
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(positions).unsqueeze(0)
        x = tok_emb + pos_emb
        x = self.dropout(x)
        
        # nn.Transformer.generate_square_subsequent_mask を使ってマスクを生成
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, dtype=torch.bool, device=x.device)

        # 自己回帰型 transformer (transformer decoder)
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        x = self.layer_norm(x)
        
        # NTP：次のトークン予測
        logits = self.fc(x)
        return logits

In [5]:
config = ModelConfig(tokenizer)
model = DNN(config).to(device)

In [6]:
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                                            Param #
DNN                                                               --
├─Embedding: 1-1                                                  2,048,000
├─Embedding: 1-2                                                  131,072
├─Dropout: 1-3                                                    --
├─TransformerEncoder: 1-4                                         --
│    └─ModuleList: 2-1                                            --
│    │    └─TransformerEncoderLayer: 3-1                          789,760
│    │    └─TransformerEncoderLayer: 3-2                          789,760
│    │    └─TransformerEncoderLayer: 3-3                          789,760
│    │    └─TransformerEncoderLayer: 3-4                          789,760
│    │    └─TransformerEncoderLayer: 3-5                          789,760
│    │    └─TransformerEncoderLayer: 3-6                          789,760
├─LayerNorm: 1-5                                        

### GPTtypeDataset
* ファイルごとに\<eod\>タグを挿入。生成時に利用する。
* 開始位置固定で文書のランダム性がなくなるが、コードがわかりやすい

### RandomGPTDataset
* GPTtypeDatasetの開始位置をランダムにしたクラス
* ステップごとに開始位置が変わるので、学習効率がよい

In [9]:
# こちらのバージョンのほうがわかりやすい
class GPTtypeDataset(Dataset):
    # (1) tokenizer.encode(text).idsがtextのID列
    def __init__(self, data_dir, tokenizer, context_size=16, stride=2):
        self.context_size = context_size
        eod_id = tokenizer.token_to_id("<eod>")

        filenames = sorted(Path(data_dir).glob("*.txt"))     # ファイル一覧
       
        # textに相当するものを作成
        all_ids = []
        for filename in filenames:
            text = filename.read_text(encoding="utf-8")
            document_ids = tokenizer.encode(text, add_special_tokens=False).ids
            all_ids.extend(document_ids)   # 文章の追記
            all_ids.append(eod_id)         # 各ファイルの末尾に<eod>

        # メモリ上にだけID列を保持する
        self.ids = torch.tensor(all_ids, dtype=torch.long)
        # (2) 各サンプルの開始位置をあらかじめ計算
        self.starts = range(0, len(self.ids) - context_size, stride)    
    
    # データ数
    def __len__(self):
        return len(self.starts)

    # (3)
    # startからcontext_sizeまでが入力データ
    # start+1からcontext_sizeまでが次のトークン予測のラベルデータ
    def __getitem__(self, idx):
        # 入力とターゲットのペアを作成
        start = self.starts[idx]
        x = self.ids[start:start + self.context_size]
        y = self.ids[start + 1:start + self.context_size + 1]
        return {"ids": x, "labels": y}

sample_dataset = GPTtypeDataset(data_dir, tokenizer, context_size=10, stride=2)
sample_dataset[0]

{'ids': tensor([3411,  593,  438,  407, 6871, 6903,  527,  265, 3068, 5911]),
 'labels': tensor([ 593,  438,  407, 6871, 6903,  527,  265, 3068, 5911, 2630])}

### 学習時に利用するタイプ
* RandomGPTDataset: GPTtypeDatasetの開始位置をランダムにしたもの
* samples_per_epochで1エポックのデータ量（開始位置の数）を変更できる

In [10]:
class RandomGPTDataset(Dataset):
    def __init__(
        self,
        data_dir,
        tokenizer,
        context_size=128,
        samples_per_epoch=10_000,
    ):
        self.context_size = context_size
        self.samples_per_epoch = samples_per_epoch
        eod_id = tokenizer.token_to_id("<eod>")
        filenames = sorted(Path(data_dir).glob("*.txt"))

        all_ids = []
        for filename in filenames:
            text = filename.read_text(encoding="utf-8")
            document_ids = tokenizer.encode(text, add_special_tokens=False).ids
            all_ids.extend(document_ids)
            all_ids.append(eod_id)

        self.ids = torch.tensor(all_ids, dtype=torch.long)

    def __len__(self):
        # 1 epochあたりに何個のランダム窓を学習するか
        # 10,000個（回？）をデフォルト値
        return self.samples_per_epoch

    def __getitem__(self, idx):
        # idxは使わず、毎回ランダムな開始位置を選ぶ
        start = torch.randint(
            low=0,
            high=len(self.ids) - self.context_size,
            size=(1,),
        ).item()

        x = self.ids[start:start + self.context_size]
        y = self.ids[start + 1:start + self.context_size + 1]

        return {"ids": x, "labels": y}

In [11]:
dataset = RandomGPTDataset(
    data_dir=data_dir,
    tokenizer=tokenizer,
    context_size=config.context_size,
    samples_per_epoch=config.samples_per_epoch,
)

dataloader = DataLoader(
    dataset=dataset,
    batch_size=config.batch_size,  # メモリ足りない場合は小さくする
    shuffle=False,   # RandomGPTDatasetの場合False　GPTtypeDatasetの場合 True
    drop_last=True,
    num_workers=0,   # ２，４の方が高速？
    pin_memory=True, # GPU使うときTrueにする
)

In [12]:
# 実行ごとに内容が変わる
# ByteLevelの時、以下を利用する
from tokenizers import decoders
tokenizer.decoder = decoders.ByteLevel()
decoded = tokenizer.decode(dataset[0]["ids"].tolist(), skip_special_tokens=False)
decoded

'普通だったら、被災して家計が苦しくなるはずだし、道路だってまだ陥没したり、ひび割れたりしている。守りに入るはずですよね。\n\n小沢:かなりかっこいいですね、その人。\n\n金子:だよね。でも、心の中では涙を流しても、それは見せないで涼しい顔をして、あえて同じ車を買い直して乗る。なかなかできることじゃない。車好きの心意気に触れて、ちょっと感激したよ。\n\n小沢:泣けてくる話ですね。BMWジャパンの社長に聞かせてやりたい。\n\n金子:その人も望んでいたけど、被災したユーザーがもう一台そのメーカーの車を購入する場合は、何らかの資金的な援助をすればいいと思うんです。助け合いだし、ブランドロイヤリティは上がるし。\n\n小沢:いいじゃないですか。同じ車を買った人は1割引き!いや2割引き!!とか。\n\n金子:行政は、被災者の被災した車の、3月11日以降の従量税と自賠責保険の還付を早く行うべきですね。\n\n小沢:あ、俺も先日急遽引っ越したんで、火災保険戻ってくるんだった。忘れてた。\n\n金子浩久blog「笑顔の向こうBehind the wheel」<eod>ランドセル素材などを製造・販売する化学メーカーの株式会社クラレが、この春小学校に入学する子ども男女とその親に「将来、就きたい職業」「就かせたい職業」のアンケートを実施しました。\n\n調査結果では、女の子の就きたい職業は「お花屋さん」が13年連続でトップの座をキープしました。2位以下には「花屋」「芸能人」と続きますが、職業の中では「教員」「看護師」など、資格や技能を生かせる職業が上位を占める結果となりました。また、女の子の親のランキングでは「看護師」「薬剤師」が1位、2位となり、今後の高齢化社会を見据えた親の思いが垣間見えます。\n\n男の子の就きたい職業トップ2はスポーツ選手で、この傾向も変わっていません。特に1位のサッカー選手は全体の6割、2位の野球選手は2割と、この2つの競技だけで全体の8割を占める結果となりました。大人から見ると野球選手の方が職業としての魅力が大きいようにも見えますが、どちらにしてもスポーツは「夢」を与えてくれる憧れの職業であることは間違いないようです。一方、'

In [13]:
optimizer = torch.optim.AdamW(model.parameters(),lr=config.learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
# step数で管理するためのデータローダー関数
def infinite_loader(dataloader):
    while True:
        for batch in dataloader:
            yield batch

data_iter = infinite_loader(dataloader)  # epochではなく、step数で計測

# ---- 学習ループの設定
# tqdm使わないほうがシンプルかも
from tqdm import tqdm               # tqdmを使い進捗状況表示をしてみた
max_iters = config.num_steps        # step数
total_token = 0                     # 累積トークン数を初期化
pbar = tqdm(range(max_iters))
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()  # flash attentionを利用の判定

# ---- ここから学習ループ
model.train()                       # trainモードを明示
for step in pbar:
    batch = next(data_iter)
    input_ids = batch["ids"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    optimizer.zero_grad()
    # GPUが対応している場合は、bf16へ変更してflash attentionを使う
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
        outputs = model(input_ids)                 # [batch, seq_len, vocab_size]
        loss = criterion(
            outputs.view(-1, config.vocab_size),   # [batch * seq_len, vocab_size]
            labels.view(-1),                       # [batch * seq_len]
        )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.max_grad_norm)  # 勾配クリップ
    optimizer.step()

    # このstepのトークン数 （バッチサイズ×系列長）を加算
    total_token += input_ids.shape[0] * input_ids.shape[1]

    # set_postfixは辞書が引数になる
    pbar.set_postfix({"loss": f"{loss.item():.4f}", "tokens": f"{total_token/1e6:.2f}M"})
    if (step+1) % 500 == 0:
        tqdm.write(f"{step+1}-step:\tloss:{loss.item():.3f}\ttokens:{total_token:,}") 


### 保存と読み込み
* 学習した重みの保存と読み込みは、最後尾についています。

### 文章生成
* 確率が一番高いトークンを選択し続けるGreedyで生成

In [16]:
@torch.inference_mode()
def generate_text(
    model,
    input_ids,
    max_new_tokens=config.seq_len,
    eos_token_id=None,
):
    model.eval()    # 評価モードへ
    generated = input_ids.to(device)
    seq_len = model.config.seq_len

    # greedyで単純に生成
    # eos_token_id　or　max_new_tokensまでで生成終了
    for _ in range(max_new_tokens):
        logits = model(generated[:, -seq_len:])                     # 単語上の確率を計算
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # 確率が最大のトークン選択
        generated = torch.cat([generated, next_token], dim=1)       # 予測idを追記
        # 終了判定
        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return generated

In [ ]:
from tokenizers import decoders
tokenizer.decoder = decoders.ByteLevel()

# 次の単語に続きやすい文を入力 乱れ入力してみた😆
prompt = "ヨーロッパ最大のフィットネスマシンメーカー"  # 異なる文章OK
#prompt = "テレビのバラエティ番組などで笑いを巻き起こすお笑い芸人" # OKコピー文を生成
#prompt = "デュアルクラッチA/T「アルファTCT」"  # 異なる文章
#prompt = "転職における市場価値をアップさせるためのスキル、知識、ノウハウ"
#prompt = "新卒市場における価値をアップさせるためのスキル、知識、ノウハウ"
#prompt = "SNSでの魅力をアップさせるためのスキルと知識"  # NG
#prompt = "図で考える力によって、問題を整理・分析"
#prompt = "昨夜の台風は10年に一度の規模だった"   # 全く異なる文章　NG

model_input = tokenizer.encode(prompt, add_special_tokens=False)
input_ids = torch.tensor([model_input.ids], dtype=torch.long, device=device)

output_ids = generate_text(
    model=model,
    input_ids=input_ids,
    max_new_tokens=256,
    eos_token_id=config.eod_token_id,
)

# 全文をデコード
print("--- 生成結果 ---")
generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)
print(generated_text)

--- 生成結果 ---
ヨーロッパ最大のフィットネスマシンメーカー「テクノジム」社は、「2010 FIFAワールドカップ南アフリカ大会」のフィットネスマシンのオフィシャル・サプライヤーに選定されました。

同大会では、テクノジム社が、出場32ヶ国中16ヶ国の各チームのベースキャンプや国際メディアセンター、そして、史上初となる審判員施設の全18ヶ所のトレーニング施設にフィットネスマシンを設置します。

テクノジムの本社があるイタリアをはじめ、フランス、ブラジル、イギリス、アルゼンチンの強豪チャンピオン候補チームのほか、開催国の南アフリカ、日本の選手たちも含め、テクノジムのマシンでトレーニングを行い、試合に挑むことになります。

各トレーニングセンターに導入される製品は以下の通りです。

・KINESIS:筋力、バランス力、柔軟性を同時に鍛えることができるマシン。サッカーにおける特定の動きのトレーニングも可能です。
・FLEXabilility:怪我の予防に欠かせないストレッチを素早く的確に行えるマシン。
・VARIO:垂直なステップ動作から大きなストライド動作まで、自由な動きが可能な新感覚のエリプ


In [ ]:
# モデルの保存
torch.save({
        "model_state_dict": model.state_dict(),
        "config": config.__dict__,  # configも一緒に保存
        }, pretrain_filename)


**モデルの読み込み方法**
* pretrain_filename: 事前学習済みのモデル
* 必要に応じて上に移動する

In [15]:
checkpoint = torch.load(pretrain_filename)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])
model = DNN(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>